In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


transform = transforms.Compose([
    transforms.ToTensor(),  
    transforms.Lambda(lambda x: x.view(-1))  
])


train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=False)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
import numpy as np


def one_hot(y, num_classes=10):
    oh = np.zeros((y.size, num_classes))
    oh[np.arange(y.size), y] = 1
    return oh

class Conv2D:
    def __init__(self, num_filters, filter_size, input_channels):
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.input_channels = input_channels
        scale = 1.0 / np.sqrt(filter_size * filter_size * input_channels)
        self.filters = np.random.randn(num_filters, input_channels, filter_size, filter_size) * scale
        self.bias = np.zeros((num_filters, 1))
    
    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        F, _, HH, WW = self.filters.shape
        out_h = H - HH + 1
        out_w = W - WW + 1
        out = np.zeros((N, F, out_h, out_w))
        
        for n in range(N):
            for f in range(F):
                for i in range(out_h):
                    for j in range(out_w):
                        region = x[n, :, i:i+HH, j:j+WW]
                        out[n, f, i, j] = np.sum(region * self.filters[f]) + self.bias[f]
        self.out = out
        return out
    
    def backward(self, d_out, lr=0.01):
        N, F, H, W = d_out.shape
        _, C, HH, WW = self.filters.shape
        dx = np.zeros_like(self.x)
        dW = np.zeros_like(self.filters)
        dB = np.zeros_like(self.bias)
        
        for n in range(N):
            for f in range(F):
                for i in range(H):
                    for j in range(W):
                        region = self.x[n, :, i:i+HH, j:j+WW]
                        dW[f] += d_out[n, f, i, j] * region
                        dx[n, :, i:i+HH, j:j+WW] += d_out[n, f, i, j] * self.filters[f]
                dB[f] += np.sum(d_out[n, f])
        
        self.filters -= lr * dW
        self.bias -= lr * dB
        return dx

class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)
    
    def backward(self, d_out):
        return d_out * (self.x > 0)

class MaxPool2x2:
    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        out = np.zeros((N, C, H//2, W//2))
        self.argmax = np.zeros_like(x, dtype=bool)
        
        for n in range(N):
            for c in range(C):
                for i in range(0, H, 2):
                    for j in range(0, W, 2):
                        region = x[n, c, i:i+2, j:j+2]
                        idx = np.unravel_index(np.argmax(region), region.shape)
                        out[n, c, i//2, j//2] = region[idx]
                        self.argmax[n, c, i+idx[0], j+idx[1]] = True
        return out
    
    def backward(self, d_out):
        dx = np.zeros_like(self.x)
        N, C, H2, W2 = d_out.shape
        for n in range(N):
            for c in range(C):
                for i in range(H2):
                    for j in range(W2):
                        dx[n, c, i*2:i*2+2, j*2:j*2+2][self.argmax[n, c, i*2:i*2+2, j*2:j*2+2]] = d_out[n, c, i, j]
        return dx

class Flatten:
    def forward(self, x):
        self.x_shape = x.shape
        return x.reshape(x.shape[0], -1)
    
    def backward(self, d_out):
        return d_out.reshape(self.x_shape)

class Dense:
    def __init__(self, in_dim, out_dim):
        scale = 1.0 / np.sqrt(in_dim)
        self.W = np.random.randn(in_dim, out_dim) * scale
        self.b = np.zeros((1, out_dim))
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, d_out, lr=0.01):
        dW = self.x.T @ d_out
        db = np.sum(d_out, axis=0, keepdims=True)
        dx = d_out @ self.W.T
        self.W -= lr * dW
        self.b -= lr * db
        return dx

class SoftmaxCrossEntropy:
    def forward(self, logits, y_true):
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)
        self.y_true = y_true
        loss = -np.mean(np.sum(y_true * np.log(self.probs + 1e-9), axis=1))
        return loss
    
    def backward(self):
        return (self.probs - self.y_true) / self.y_true.shape[0]

In [9]:

conv = Conv2D(8, 3, 1)
relu = ReLU()
pool = MaxPool2x2()
flat = Flatten()
fc = Dense(13*13*8, 10)
loss_fn = SoftmaxCrossEntropy()

epochs = 3
lr = 0.01
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (xb, yb) in enumerate(train_loader):
    
        xb = xb.numpy().reshape(-1,1,28,28)
        yb = yb.numpy()
        yb_oh = one_hot(yb)

   
        out = conv.forward(xb)
        out = relu.forward(out)
        out = pool.forward(out)
        out = flat.forward(out)
        logits = fc.forward(out)
        loss = loss_fn.forward(logits, yb_oh)
        total_loss += loss

 
        d_out = loss_fn.backward()
        d_out = fc.backward(d_out, lr)
        d_out = flat.backward(d_out)
        d_out = pool.backward(d_out)
        d_out = relu.backward(d_out)
        conv.backward(d_out, lr)

      
        if (batch_idx+1) % 10 == 0:  # every 10 batches
            print(f"Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)}, Loss={loss:.4f}")

    print(f"Epoch {epoch+1} finished, Total Loss={total_loss:.4f}")


C:\Users\sshak\AppData\Local\Temp\ipykernel_11396\3056830825.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  out[n, f, i, j] = np.sum(region * self.filters[f]) + self.bias[f]


Epoch 1, Batch 10/7500, Loss=2.2920
Epoch 1, Batch 20/7500, Loss=2.1454
Epoch 1, Batch 30/7500, Loss=1.8679
Epoch 1, Batch 40/7500, Loss=1.7633
Epoch 1, Batch 50/7500, Loss=1.7922
Epoch 1, Batch 60/7500, Loss=1.3088
Epoch 1, Batch 70/7500, Loss=1.4684
Epoch 1, Batch 80/7500, Loss=1.6387
Epoch 1, Batch 90/7500, Loss=1.1248
Epoch 1, Batch 100/7500, Loss=1.1853
Epoch 1, Batch 110/7500, Loss=1.0392
Epoch 1, Batch 120/7500, Loss=1.4069
Epoch 1, Batch 130/7500, Loss=1.1357
Epoch 1, Batch 140/7500, Loss=0.7601
Epoch 1, Batch 150/7500, Loss=1.0549
Epoch 1, Batch 160/7500, Loss=0.7164
Epoch 1, Batch 170/7500, Loss=0.5820
Epoch 1, Batch 180/7500, Loss=0.6584
Epoch 1, Batch 190/7500, Loss=0.9384
Epoch 1, Batch 200/7500, Loss=0.8441
Epoch 1, Batch 210/7500, Loss=0.7343
Epoch 1, Batch 220/7500, Loss=0.9280
Epoch 1, Batch 230/7500, Loss=0.6480
Epoch 1, Batch 240/7500, Loss=0.7615
Epoch 1, Batch 250/7500, Loss=0.7296
Epoch 1, Batch 260/7500, Loss=0.4125
Epoch 1, Batch 270/7500, Loss=0.5791
Epoch 1, B

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ---------------- Device ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- Data ----------------
transform = transforms.Compose([
    transforms.ToTensor(),  # normalize to [0,1]
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ---------------- Model ----------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)  # 1 channel -> 8 filters
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(8*13*13, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

# ---------------- Loss & Optimizer ----------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------- Training ----------------
epochs = 3
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        
    print(f"Epoch {epoch+1} finished, Avg Loss={total_loss/len(train_loader):.4f}")

# ---------------- Testing ----------------


Epoch 1 finished, Avg Loss=0.2018
Epoch 2 finished, Avg Loss=0.0670
Epoch 3 finished, Avg Loss=0.0436
Test Accuracy: 0.9831


In [12]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f"Test Accuracy: {correct/total:.4f}")

Test Accuracy: 0.9831
